# How a Chunk Becomes a 384-Float Vector

This notebook walks through the 4 steps of the embedding pipeline using **real computed values**.

Model: `all-MiniLM-L6-v2` via sentence-transformers  
Example chunk: *"pembrolizumab improves survival in MSI-H tumors"*

---
**Steps covered:**
1. Tokenization — text → token IDs
2. Initial embeddings — shape of the raw token matrix
3. Final embedding — shape after mean pooling
4. Cosine similarity — why semantically close chunks produce close vectors

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Model loaded: {MODEL_NAME}")
print(f"Embedding dimensions: {model.get_sentence_embedding_dimension()}")

## Step 1 — Tokenization

The tokenizer splits text into subword tokens. Rare or long words (like drug names) get split into multiple pieces.

In [ ]:
chunk = "pembrolizumab improves survival in MSI-H tumors"

encoded = tokenizer(chunk, return_tensors="pt")
token_ids = encoded["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(token_ids)

print(f"Input text : {chunk}")
print(f"Token count: {len(tokens)}")
print()
print(f"{'Token':<15} {'ID':>6}")
print("-" * 23)
for tok, id_ in zip(tokens, token_ids):
    print(f"{tok:<15} {id_:>6}")

**What to notice:**
- `[CLS]` and `[SEP]` are special tokens added automatically by the tokenizer
- `pembrolizumab` splits into multiple `##` prefixed subwords — the `##` means "continuation of previous token"
- Common words like `improves`, `in`, `tumors` are single tokens
- Each token maps to a unique integer ID (the lookup table key in Step 2)

## Step 2 — Token Embedding Matrix (Initial, Context-Free)

Before attention, each token ID is mapped to a 384-float vector via a learned lookup table.  
The result is a matrix of shape `(num_tokens, 384)` — one row per token, 384 floats per row.

In [ ]:
import torch

# Get the embedding lookup table (word embeddings layer)
word_embeddings = model[0].auto_model.embeddings.word_embeddings

input_ids = encoded["input_ids"]  # shape: (1, num_tokens)
token_embeds = word_embeddings(input_ids)  # shape: (1, num_tokens, 384)
token_embeds = token_embeds.squeeze(0)  # shape: (num_tokens, 384)

print(f"Token embedding matrix shape: {tuple(token_embeds.shape)}")
print(f"  = {token_embeds.shape[0]} tokens × {token_embeds.shape[1]} dimensions")
print()
print("First 6 floats of each token's initial vector (context-free):")
print(f"{'Token':<15} {'First 6 values'}")
print("-" * 60)
with torch.no_grad():
    for tok, vec in zip(tokens, token_embeds):
        preview = vec[:6].detach().numpy()
        formatted = "  ".join(f"{v:+.3f}" for v in preview)
        print(f"{tok:<15} [{formatted} ...]")

**What to notice:**
- Every token has a unique 384-float vector — but at this stage, `survival` would have the SAME vector whether it appears in an oncology paper or a nature documentary
- The `##liz`, `##um`, `##ab` subwords of `pembrolizumab` each have their own initial vectors
- This is the INPUT to the 6 transformer layers — context is added next

## Step 3 — Final Embedding After Self-Attention + Mean Pooling

The model runs 6 transformer layers (self-attention), then mean-pools all token vectors into a single `(384,)` vector.

In [ ]:
embedding = model.encode(chunk)

print(f"Final embedding shape: {embedding.shape}")
print(f"dtype: {embedding.dtype}")
print()
print(f"First 10 values : {embedding[:10].round(4)}")
print(f"Last 10 values  : {embedding[-10:].round(4)}")
print()
print(f"Min  : {embedding.min():.4f}")
print(f"Max  : {embedding.max():.4f}")
print(f"Mean : {embedding.mean():.4f}")
print(f"Norm : {np.linalg.norm(embedding):.4f}")

**What to notice:**
- Shape is `(384,)` — one vector for the entire chunk, regardless of how many tokens it had
- Values are small floats (typically between -1 and +1 after normalization)
- The norm (~1.0) means the vector is L2-normalized — this is what makes cosine similarity just a dot product

## Step 4 — Cosine Similarity: Why Semantic Search Works

Embed three sentences and compare how close they are in vector space.

In [ ]:
sentences = [
    "pembrolizumab improves survival in MSI-H tumors",  # our original chunk
    "immunotherapy response in mismatch repair deficient cancer",  # same domain, different words
    "interest rates and bond yields in Q3 2024",  # completely unrelated
]

embeddings = model.encode(sentences)
print(f"Embeddings matrix shape: {embeddings.shape}")
print(f"  = {embeddings.shape[0]} sentences × {embeddings.shape[1]} dimensions")
print()

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


sim_AB = cosine_similarity(embeddings[0], embeddings[1])
sim_AC = cosine_similarity(embeddings[0], embeddings[2])
sim_BC = cosine_similarity(embeddings[1], embeddings[2])

labels = [
    ("Chunk vs Same-domain sentence", sim_AB),
    ("Chunk vs Unrelated sentence", sim_AC),
    ("Same-domain vs Unrelated", sim_BC),
]

print(f"{'Pair':<35} {'Cosine Similarity':>18}  Interpretation")
print("-" * 80)
for label, sim in labels:
    bar = "█" * int(sim * 30)
    interp = "CLOSE" if sim > 0.5 else ("MODERATE" if sim > 0.2 else "FAR")
    print(f"{label:<35} {sim:>18.4f}  {bar} {interp}")

**What to notice:**
- The oncology chunk and the same-domain sentence (using completely different words!) score high similarity — the model captures *meaning*, not keyword overlap
- The finance sentence scores near zero — semantically in a different part of vector space
- This is exactly what Day 7 (retrieval) exploits: embed a user's query and find the chunks with the highest cosine similarity

## Step 5 — Visualize the Similarity Matrix

In [ ]:
import matplotlib.pyplot as plt

n = len(sentences)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(embeddings[i], embeddings[j])

short_labels = [
    "pembrolizumab /\nMSI-H tumors",
    "immunotherapy /\nMMR deficient",
    "interest rates /\nbond yields",
]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0, vmax=1)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_yticklabels(short_labels, fontsize=10)

for i in range(n):
    for j in range(n):
        val = sim_matrix[i, j]
        color = "black" if val > 0.4 else "white"
        ax.text(
            j,
            i,
            f"{val:.3f}",
            ha="center",
            va="center",
            fontsize=12,
            color=color,
            fontweight="bold",
        )

plt.colorbar(im, ax=ax, label="Cosine Similarity")
ax.set_title(
    "Semantic Similarity Between Sentences\n(diagonal = 1.0, self-similarity)", fontsize=12
)
plt.tight_layout()
plt.savefig("similarity_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: similarity_matrix.png")

## Bonus — Embed a Batch (What embed.py Will Do)

On Day 5 we'll call `model.encode()` on all 1,514 chunks at once.  
Here's a preview of the output shape.

In [ ]:
batch = [
    "pembrolizumab improves survival in MSI-H tumors",
    "TC score greater than 20% is required for NGS eligibility",
    "BRCA1 mutation increases lifetime risk of breast cancer",
    "neoadjuvant chemotherapy reduces tumor burden before surgery",
    "HER2 overexpression is a biomarker for trastuzumab response",
]

batch_embeddings = model.encode(batch, show_progress_bar=False)

print(f"Input: {len(batch)} chunks")
print(f"Output shape: {batch_embeddings.shape}")
print(
    f"  = {batch_embeddings.shape[0]} rows (chunks) × {batch_embeddings.shape[1]} cols (dimensions)"
)
print()
print("This is what embed.py will produce for all 1,514 chunks:")
print("  Expected shape: (1514, 384)")
print(f"  Memory: ~{1514 * 384 * 4 / 1024:.0f} KB (~{1514 * 384 * 4 / 1024 / 1024:.1f} MB)")

---
## Summary

| Step | Input | Output | Key insight |
|------|-------|--------|-------------|
| Tokenization | text string | list of token IDs | rare words split into subwords |
| Lookup table | token IDs | `(N, 384)` matrix | context-free initial vectors |
| Self-attention × 6 | `(N, 384)` | `(N, 384)` | each token absorbs neighbor context |
| Mean pooling | `(N, 384)` | `(384,)` | one vector captures the whole chunk |

Next: **Day 5 build** — `src/pubmed_rag/embed.py` applies this to all 1,514 chunks from `chunk.py`.